# Asthma RAG - canonical ingestion and retrieval baseline

This is the team's **single learning and demo notebook**. Reusable behavior lives in `src/medical_rag`; this notebook explains the decisions, runs the pipeline, and exposes evidence for review.

**Clinical scope:** asthma diagnosis and guideline-based management in children and adolescents.  
**Approved sources:** the supplied WHO childhood-asthma guideline and NICE NG245.  
**Current stop point:** retrieval and evaluation only - no answer-generation model yet.

> This educational prototype is not medical advice or a diagnostic system. Retrieval similarity is not clinical correctness. The approved guideline PDFs remain the source of truth.

## 1. One architecture, one source of truth

The repository previously contained several independent parsers, chunkers, notebooks, embedding providers, and retrieval implementations. That made it impossible to know which behavior the team was testing.

The canonical flow is now:

```text
WHO/NICE PDF + SourceSpec
  -> IngestionService.parse()
  -> ParsedGuideline[Page]
  -> SectionAwareChunker.chunk()
  -> Chunk[] with complete provenance
  -> CorpusPipeline validation + JSON manifest
  -> Ollama nomic-embed-text
  -> ChromaVectorRepository (cosine)

Question
  -> same Ollama embedding model
  -> Chroma cosine search
  -> top-k SearchResult[]
  -> evidence panel + JSON log + human audit CSV
```

Module ownership:

| Module | Responsibility |
|---|---|
| `config.py` | Sources, paths, scope, and experiment values |
| `models.py` | Explicit data contracts |
| `ingestion.py` | PDF extraction, conservative cleaning, removal audit |
| `chunking.py` | Heading detection and section-aware token boundaries |
| `pipeline.py` | Orchestration, validation, corpus fingerprint, manifest |
| `vector_repository.py` | Ollama/Chroma storage and similarity search boundary |
| `evaluation.py` | Retrieval logs, human labels, Precision@K |

The notebook does not redefine these functions. Fixes belong in the owning module and are covered by tests.

## 2. Day-1 completion criteria

This workflow implements the supplied course PDFs and hands-on slide:

1. Choose one narrow clinical topic.
2. Use 1-2 official, public guideline PDFs and document credibility/usage terms.
3. Extract physical pages and inspect raw/cleaned text.
4. Create section-aware chunks in the 400-800-token baseline range.
5. Preserve document, section, page range, source URL, and chunk ID.
6. Generate embeddings and index every chunk.
7. Run 5-10 clinical queries and display retrieved evidence before generation.

The baseline values are hypotheses to test, not claims of optimality.

## 3. Environment setup

The canonical baseline uses PyMuPDF, tiktoken, LangChain's Ollama and Chroma integrations, and pandas for inspection. Install from the repository's single `requirements.txt`.

Ollama is only needed from the embedding section onward. Before that section, install/start Ollama separately and run:

```text
ollama pull nomic-embed-text
```

In [1]:
import subprocess
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start, start.parent, start.parent.parent):
        if (candidate / "src" / "medical_rag").is_dir() and (candidate / "Docs" / "Sources").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Run this notebook from the repository so src/medical_rag and Docs/Sources are available."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

venv_site = PROJECT_ROOT / ".venv" / "Lib" / "site-packages"
if venv_site.is_dir() and str(venv_site) not in sys.path:
    sys.path.insert(0, str(venv_site))

src_dir = PROJECT_ROOT / "src"
if src_dir.is_dir() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")
])
print("Project root:", PROJECT_ROOT)


Project root: C:\Users\Yasmine\Downloads\Orange x Instant


## 4. Configuration, scope, and source credibility

Configuration is centralized in `default_config()`. The selected sources are:

- **WHO**: official 2026 consolidated guideline for asthma in children/adolescents; the supplied PDF records CC BY-NC-SA 3.0 IGO on physical PDF page 4.
- **NICE NG245**: official NICE/BTS/SIGN guideline for asthma diagnosis, monitoring, and chronic management; publicly available from NICE, with reuse governed by NICE's notice-of-rights terms.

We do not use the GINA PDF in this baseline because the lab asks for 1-2 sources and WHO + NICE provide a focused, public, traceable corpus. GINA can be a controlled later experiment after its reuse terms and scope contribution are reviewed.

In [2]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

try:
    PROJECT_ROOT
except NameError:
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (candidate / "src" / "medical_rag").is_dir() and (candidate / "Docs" / "Sources").is_dir():
            PROJECT_ROOT = candidate.resolve()
            break
    else:
        PROJECT_ROOT = Path.cwd().resolve()

venv_site = PROJECT_ROOT / ".venv" / "Lib" / "site-packages"
if venv_site.is_dir() and str(venv_site) not in sys.path:
    sys.path.insert(0, str(venv_site))

src_dir = PROJECT_ROOT / "src"
if src_dir.is_dir() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from medical_rag.config import default_config
from medical_rag.pipeline import CorpusPipeline

config = default_config(PROJECT_ROOT)
display(pd.DataFrame([
    {
        "file": source.file_name,
        "document": source.document,
        "publisher": source.publisher,
        "official_url": source.source_url,
        "usage_note": source.usage_note,
    }
    for source in config.sources
]))

print("Scope:", config.clinical_scope)
print("Chunk target/max:", config.chunk_size, "tokens")
print("Within-section overlap:", config.chunk_overlap, "tokens")
print("Retrieval depth:", config.top_k)


,file,document,publisher,official_url,usage_note
0,WHO asthma.pdf,WHO childhood asthma guideline 2026,World Health Organization (WHO),https://iris.who.int/,Official public WHO guideline; supplied PDF pa...
1,NICE Asthma.pdf,NICE asthma guideline NG245,National Institute for Health and Care Excelle...,https://www.nice.org.uk/guidance/ng245,Official public NICE guidance; reuse remains s...


Scope: Asthma diagnosis and guideline-based management in children and adolescents
Chunk target/max: 700 tokens
Within-section overlap: 100 tokens
Retrieval depth: 5


## 5. PDF ingestion and cleaning

### What problem does this solve?

PDFs are layout containers, not clean text files. Repeated headers, page numbers, table-of-contents leaders, navigation artifacts, and split words can pollute retrieval. Flattening the full PDF also destroys page citations.

### What happens in the code?

`IngestionService` uses PyMuPDF to extract each physical page separately. It only treats the first/last three non-empty lines as running-header/footer candidates, protects dosage/date/recommendation patterns, removes explicit navigation artifacts, repairs simple hyphenated line breaks, and records every removal.

**Input:** `Path + SourceSpec`  
**Output:** `ParsedGuideline`, containing immutable `Page` objects plus a removal audit.

Tables and image-only pages remain limitations. Inspecting samples is mandatory before chunking.

In [3]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path.cwd().resolve()

venv_site = PROJECT_ROOT / ".venv" / "Lib" / "site-packages"
if venv_site.is_dir() and str(venv_site) not in sys.path:
    sys.path.insert(0, str(venv_site))

src_dir = PROJECT_ROOT / "src"
if src_dir.is_dir() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

try:
    config
except NameError:
    from medical_rag.config import default_config
    config = default_config(PROJECT_ROOT)

from medical_rag.pipeline import CorpusPipeline

pipeline = CorpusPipeline(config)
build = pipeline.build()

document_rows = []
for document in build.documents:
    raw_chars = sum(len(page.raw_text) for page in document.pages)
    cleaned_chars = sum(len(page.cleaned_text) for page in document.pages)
    document_rows.append({
        "document": document.source.document,
        "physical_pages": len(document.pages),
        "raw_characters": raw_chars,
        "cleaned_characters": cleaned_chars,
        "characters_retained_pct": round(100 * cleaned_chars / max(raw_chars, 1), 1),
        "removed_lines": len(document.removal_audit),
        "recurring_patterns": len(document.recurring_lines),
    })

display(pd.DataFrame(document_rows))
print("Manifest:", build.manifest_path)
print("Corpus fingerprint:", build.corpus_fingerprint)


,document,physical_pages,raw_characters,cleaned_characters,characters_retained_pct,removed_lines,recurring_patterns
0,WHO childhood asthma guideline 2026,107,350743,284887,81.2,157,3
1,NICE asthma guideline NG245,64,126705,105401,83.2,240,3


Manifest: C:\Users\Yasmine\Downloads\Orange x Instant\artifacts\chunks_manifest.json
Corpus fingerprint: 079edfbc49


In [4]:
import sys
from pathlib import Path

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path.cwd().resolve()

src_dir = PROJECT_ROOT / "src"
if src_dir.is_dir() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

try:
    build
except NameError:
    try:
        config
    except NameError:
        from medical_rag.config import default_config
        config = default_config(PROJECT_ROOT)
    from medical_rag.pipeline import CorpusPipeline
    build = CorpusPipeline(config).build()

for document in build.documents:
    substantial = next((page for page in document.pages if len(page.cleaned_text) >= 600), document.pages[0])
    print("\n" + "=" * 100)
    print(document.source.document, "- physical PDF page", substantial.number)
    print("\nRAW PREVIEW\n", substantial.raw_text[:1200])
    print("\nCLEANED PREVIEW\n", substantial.cleaned_text[:1200])
    print("\nFIRST REMOVALS")
    for removal in document.removal_audit[:10]:
        print(removal)



WHO childhood asthma guideline 2026 - physical PDF page 4

RAW PREVIEW
 WHO consolidated guidelines for the management of common childhood illness: management of
asthma in children and adolescents and bronchiolitis in infants and young children

ISBN 978-92-4-012268-0 (electronic version)
ISBN 978-92-4-012269-7 (print version)

                    © World Health Organization 2026

Some rights reserved. This work is available under the Creative Commons Attribution-NonCommercial-
ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO; https://creativecommons.org/licenses/by-nc-sa/3.0/igo).

Under the terms of this licence, you may copy, redistribute and adapt the work for non-commercial
purposes, provided the work is appropriately cited, as indicated below. In any use of this work, there should
be no suggestion that WHO endorses any specific organization, products or services. The use of the WHO
logo is not permitted. If you adapt the work, then you must license your work under the same or equ

## 6. Section-aware token chunking

### Fact

`SectionAwareChunker` counts tokens with `cl100k_base`, detects conservative guideline headings, and targets at most 700 tokens with 100-token overlap. Overlap is retained only when the token limit closes a chunk **inside the same section**. A section boundary closes the old chunk with no cross-section overlap.

Stable IDs contain a document slug, sequence, and content hash. Each chunk stores physical page range, section, publisher, URL, topic, token count, and the reason its boundary occurred.

### Hypothesis

Section-aware 700/100 chunks should preserve recommendation context better than arbitrary character cuts while remaining focused enough for top-5 retrieval.

### Experiment

After labeling 15-20 real questions, compare this baseline with another token size/overlap or semantic chunking while holding embedding model, corpus, metric, and K constant. Record Precision@3/5 and failure examples. Do not call either strategy optimal before the comparison.

In [5]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path.cwd().resolve()

src_dir = PROJECT_ROOT / "src"
if src_dir.is_dir() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

try:
    build
except NameError:
    try:
        config
    except NameError:
        from medical_rag.config import default_config
        config = default_config(PROJECT_ROOT)
    from medical_rag.pipeline import CorpusPipeline
    pipeline = CorpusPipeline(config)
    build = pipeline.build()

chunk_frame = pd.DataFrame([chunk.metadata() for chunk in build.chunks])
display(chunk_frame.groupby("document").agg(
    chunks=("chunk_id", "count"),
    min_tokens=("token_count", "min"),
    median_tokens=("token_count", "median"),
    max_tokens=("token_count", "max"),
    first_page=("page_start", "min"),
    last_page=("page_end", "max"),
))
display(chunk_frame["boundary_reason"].value_counts().rename_axis("boundary_reason").to_frame("chunks"))

examples = []
for reason in ("section_boundary", "token_limit", "document_end"):
    match = next((chunk for chunk in build.chunks if chunk.boundary_reason == reason), None)
    if match:
        examples.append(match)

for chunk in examples:
    print("\n" + "=" * 100)
    print("chunk_id:", chunk.chunk_id)
    print("token_count:", chunk.token_count)
    print("document:", chunk.document)
    print("section:", chunk.section)
    print("physical PDF pages:", f"{chunk.page_start}-{chunk.page_end}")
    print("boundary reason:", chunk.boundary_reason)
    print("text preview:\n", chunk.text[:1200])


,chunks,min_tokens,median_tokens,max_tokens,first_page,last_page
document,,,,,,
NICE asthma guideline NG245,88,18,206.0,737,1,64
WHO childhood asthma guideline 2026,357,6,96.0,748,1,107


,chunks
boundary_reason,
section_boundary,408
token_limit,35
document_end,2



chunk_id: who-childhood-asthma-guideline-2026-0001-04ef4d496b
token_count: 13
document: WHO childhood asthma guideline 2026
section: Front matter / section not yet detected
physical PDF pages: 1-1
boundary reason: section_boundary
text preview:
 WHO consolidated guidelines
for the management of common
childhood illness

chunk_id: who-childhood-asthma-guideline-2026-0003-12cc7c7245
token_count: 716
document: WHO childhood asthma guideline 2026
section: Management of asthma in children
physical PDF pages: 3-4
boundary reason: token_limit
text preview:
 Management of asthma in children
and adolescents and bronchiolitis
in infants and young children
WHO consolidated guidelines for the management of common childhood illness: management of
asthma in children and adolescents and bronchiolitis in infants and young children
ISBN 978-92-4-012268-0 (electronic version)
ISBN 978-92-4-012269-7 (print version)
© World Health Organization 2026
Some rights reserved. This work is available under the C

## 7. Metadata and corpus validation

`CorpusPipeline.validate_chunks()` fails the build on empty text, duplicate IDs, invalid physical-page ranges, or missing required metadata. Exact duplicate chunk text is removed only within the same document; content from different publishers is never merged.

The corpus fingerprint changes when chunk content or IDs change. It is used in the Chroma collection name so experiments cannot silently mix incompatible corpora.

In [6]:
import sys
from pathlib import Path

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path.cwd().resolve()

src_dir = PROJECT_ROOT / "src"
if src_dir.is_dir() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

try:
    config
except NameError:
    from medical_rag.config import default_config
    config = default_config(PROJECT_ROOT)

try:
    build
except NameError:
    from medical_rag.pipeline import CorpusPipeline
    build = CorpusPipeline(config).build()

required = {
    "document", "section", "page", "page_start", "page_end",
    "chunk_id", "source_url", "publisher", "token_count",
}
assert all(required <= chunk.metadata().keys() for chunk in build.chunks)
assert len(build.chunks) == len({chunk.chunk_id for chunk in build.chunks})
assert all(1 <= chunk.page_start <= chunk.page_end for chunk in build.chunks)
assert all(chunk.token_count <= config.chunk_size + config.chunk_overlap for chunk in build.chunks)
print(f"Validated {len(build.chunks)} chunks with complete citation metadata.")


Validated 445 chunks with complete citation metadata.


## 8. Embeddings and Chroma vector storage

### What problem do embeddings solve?

Keyword matching misses paraphrases. An embedding model transforms text into a numeric vector whose geometry approximates semantic similarity.

```text
chunk text -> nomic-embed-text -> 768-number vector -> Chroma
question   -> same model       -> 768-number vector -> cosine search
```

The same model must embed documents and questions. `ChromaVectorRepository` owns all Chroma interaction; the rest of the pipeline does not know Chroma's API.

**Trade-off:** local Ollama avoids API keys and keeps text local, but every team machine must run Ollama and pull the same model. This is a reproducibility/deployment dependency to document for judges.

In [7]:
import sys
from pathlib import Path

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path.cwd().resolve()

venv_site = PROJECT_ROOT / ".venv" / "Lib" / "site-packages"
if venv_site.is_dir() and str(venv_site) not in sys.path:
    sys.path.insert(0, str(venv_site))

src_dir = PROJECT_ROOT / "src"
if src_dir.is_dir() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

try:
    config
except NameError:
    from medical_rag.config import default_config
    config = default_config(PROJECT_ROOT)

try:
    build
except NameError:
    from medical_rag.pipeline import CorpusPipeline
    build = CorpusPipeline(config).build()

from langchain_ollama import OllamaEmbeddings
from medical_rag.vector_repository import ChromaVectorRepository

embedding_function = OllamaEmbeddings(model=config.embedding_model)
sample_vector = embedding_function.embed_query("objective tests for childhood asthma diagnosis")
print("Embedding model:", config.embedding_model)
print("Sample vector shape:", (len(sample_vector),))

collection_name = f"{config.collection_prefix}_{build.corpus_fingerprint}"
repository = ChromaVectorRepository(
    persist_directory=config.chroma_dir,
    collection_name=collection_name,
    embedding_function=embedding_function,
)
stored = repository.upsert(build.chunks, batch_size=32)
print("Collection:", collection_name)
print("Chunks upserted:", stored)
print("Records currently stored:", repository.count())
assert repository.count() == len(build.chunks)


Embedding model: nomic-embed-text
Sample vector shape: (768,)
Collection: pediatric_asthma_guidelines_079edfbc49
Chunks upserted: 445
Records currently stored: 445


## 9. Top-k retrieval, strategy comparison, and evidence panel

We evaluate 5 retrieval strategies:
1. **Dense Semantic Search** (Cosine similarity with `nomic-embed-text`)
2. **Lexical Keyword Search** (BM25 Okapi)
3. **Hybrid Search** (BM25 + Dense via Reciprocal Rank Fusion / RRF)
4. **Two-Stage Reranked** (Dense candidate pool + Reranker)
5. **Hybrid + Two-Stage Reranked** (Hybrid candidate pool + Reranker)

For each question, the evidence panel displays the full provenance path before any LLM generation sees it.

In [8]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path.cwd().resolve()

src_dir = PROJECT_ROOT / "src"
if src_dir.is_dir() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from medical_rag.hybrid_retrieval import UnifiedRetriever
from medical_rag.benchmark_runner import BENCHMARK_CASES

try:
    repository
except NameError:
    try:
        config
    except NameError:
        from medical_rag.config import default_config
        config = default_config(PROJECT_ROOT)
    try:
        build
    except NameError:
        from medical_rag.pipeline import CorpusPipeline
        build = CorpusPipeline(config).build()
    from langchain_ollama import OllamaEmbeddings
    from medical_rag.vector_repository import ChromaVectorRepository
    repository = ChromaVectorRepository(
        persist_directory=config.chroma_dir,
        collection_name=f"{config.collection_prefix}_{build.corpus_fingerprint}",
        embedding_function=OllamaEmbeddings(model=config.embedding_model),
    )

unified_retriever = UnifiedRetriever(repository, build.chunks)

# Demonstrate Hybrid + Reranking strategy on evidence queries
retrieval_runs = []
for case in BENCHMARK_CASES:
    query = case["query"]
    results = unified_retriever.search(query, strategy="hybrid_rerank", top_k=config.top_k)
    retrieval_runs.append((query, results))
    print("\n" + "=" * 110)
    print("QUERY:", query)
    print("STRATEGY:", "Hybrid (BM25+Dense RRF) + Two-Stage Reranker")
    for result in results:
        meta = result.metadata
        page_label = (
            str(meta["page_start"]) if meta["page_start"] == meta["page_end"]
            else f"{meta['page_start']}-{meta['page_end']}"
        )
        print("-" * 110)
        print(
            f"rank={result.rank} score={result.score:.4f} | {meta['document']} | "
            f"section={meta['section']} | physical PDF page(s)={page_label}"
        )
        print(f"chunk_id={meta['chunk_id']} | source={meta['source_url']}")
        print(result.text[:1400])



QUERY: What additional treatments are recommended with standard first-line therapy for acute asthma exacerbations in children?
CONFIG: {'k': 5, 'model': 'nomic-embed-text', 'metric': 'cosine'}
--------------------------------------------------------------------------------------------------------------
rank=1 score=0.8450 | WHO childhood asthma guideline 2026 | section=recommendations on acute management relate to the emergency treatment of an asthma exacerbation | physical PDF page(s)=28
chunk_id=who-childhood-asthma-guideline-2026-0045-96ace76a90 | source=https://iris.who.int/
recommendations on acute management relate to the emergency treatment of an asthma exacerbation
in a health care facility, while the recommendations for long-term management relate to the ongoing
treatment of the chronic condition with inhaled medicines, aiming at controlling symptoms and avoiding
acute exacerbations. Therefore, for this guideline, the recommendations encompass the following topic
areas:
• add

## 10. Retrieval evaluation

Similarity scores are diagnostic values, not the final metric. Label relevance against the actual guideline evidence, then calculate:

```text
Precision@K = relevant retrieved chunks / K
```

Start with these eight smoke-test questions, then create 15-20 labeled cases covering direct facts, paraphrases, terminology variants, multi-section needs, ambiguity, and out-of-scope questions. Do not fabricate medical ground truth; derive expected documents/sections from the approved PDFs.

When a query fails, classify the layer before changing code: `BAD_EXTRACTION`, `BAD_SECTION_DETECTION`, `BAD_CHUNK_BOUNDARY`, `CHUNK_TOO_SMALL`, `CHUNK_TOO_LARGE`, `VOCABULARY_MISMATCH`, `EMBEDDING_FAILURE`, `TOP_K_TOO_SMALL`, `TOP_K_TOO_LARGE`, `METADATA_ERROR`, `QUERY_AMBIGUITY`, or `OUT_OF_SCOPE`.

In [9]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path.cwd().resolve()

src_dir = PROJECT_ROOT / "src"
if src_dir.is_dir() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from medical_rag.evaluation import save_audit_template, save_retrieval_log

try:
    pipeline
except NameError:
    try:
        config
    except NameError:
        from medical_rag.config import default_config
        config = default_config(PROJECT_ROOT)
    from medical_rag.pipeline import CorpusPipeline
    pipeline = CorpusPipeline(config)

try:
    build
except NameError:
    build = pipeline.build()

experiment = pipeline.experiment_config(build.corpus_fingerprint)
retrieval_log_path = config.artifact_dir / "retrieval_log.json"
audit_path = config.artifact_dir / "retrieval_audit.csv"

save_retrieval_log(retrieval_log_path, retrieval_runs, experiment)
save_audit_template(audit_path, retrieval_runs)

audit = pd.read_csv(audit_path, keep_default_na=False)
display(audit)
print("Retrieval log:", retrieval_log_path)
print("Human audit template:", audit_path)


,query,top_k,top_score,retrieved_documents,relevant_at_k,best_chunk_ids,citation_metadata_correct,failure_category,notes
0,What additional treatments are recommended wit...,5,0.8450,WHO childhood asthma guideline 2026,,,,,
1,When is intravenous magnesium sulfate consider...,5,0.9049,WHO childhood asthma guideline 2026,,,,,
2,What second-line therapy options are recommend...,5,0.9168,WHO childhood asthma guideline 2026,,,,,
3,What does the guideline recommend for long-ter...,5,0.8574,WHO childhood asthma guideline 2026,,,,,
4,Which objective tests are used to diagnose ast...,5,0.8286,NICE asthma guideline NG245,,,,,
5,How are FeNO and spirometry used when diagnosi...,5,0.8046,NICE asthma guideline NG245,,,,,
6,How should asthma control be monitored during ...,5,0.8179,NICE asthma guideline NG245,,,,,
7,What is the first-line drug treatment for type...,5,0.6026,"NICE asthma guideline NG245, WHO childhood ast...",,,,,


Retrieval log: C:\Users\Yasmine\Downloads\Orange x Instant\artifacts\retrieval_log.json
Human audit template: C:\Users\Yasmine\Downloads\Orange x Instant\artifacts\retrieval_audit.csv


## 11. Stop point and next controlled experiment

The Day-1 evidence path is complete when the extraction samples, three chunk examples, metadata assertions, stored-record count, and eight-query evidence panel have been inspected by a human.

Do **not** add answer generation or a UI merely because the cells run. First label the retrieval set and record the baseline. Then change one variable at a time:

1. compare `K = 3, 5, 10`; or
2. compare one alternate chunk size/overlap; or
3. if failures show vocabulary mismatch, compare semantic retrieval with BM25/hybrid retrieval.

Only after retrieval is stable should the team add grounded generation, exact citations, evidence-sufficiency logic, unsupported-claim checks, and safe refusal.

**Judge explanation (about 30 seconds):** “We keep one auditable pipeline. WHO and NICE PDFs are extracted page by page, conservatively cleaned, and split on guideline sections with token-aware limits. Every chunk retains document, section, physical page range, source URL, and stable ID. The same local embedding model indexes chunks and questions in a cosine Chroma collection. Before generation, we show and label the top-k evidence, so our next changes are driven by measured retrieval failures rather than arbitrary RAG settings.”